# The 5.5 Cliff: reproducing DeepMind's two softmax bugs

Paper: Barron, Gnanapragasam & Tschannen (Google DeepMind), "A Fix for Softmax and Cross-Entropy", arXiv:2610.31642 (22 Oct 2026). Every standard softmax / log-softmax / cross-entropy in JAX, PyTorch and TensorFlow carries both bugs. This notebook reproduces them in NumPy — no framework needed, because the bugs live in the floating-point arithmetic itself.

In [ ]:
import math
import numpy as np

def bf16(x):
    """Round a float to bfloat16 (8 mantissa bits) via bit truncation."""
    a = np.array([x], dtype=np.float32)
    v = (a.view(np.uint32) + 0x8000) & 0xFFFF0000
    return float(v.view(np.float32)[0])

def std_target_grad_fp32(m):
    """Standard target gradient q_t - 1, with q_t = 1 / (1 + exp(-m)) in fp32."""
    e = np.float32(np.exp(-np.float32(m)))
    q = np.float32(1.0) / (np.float32(1.0) + e)
    return float(q - np.float32(1.0))

def std_target_grad_bf16(m):
    e = bf16(math.exp(-bf16(m)))
    return bf16(1.0 / bf16(1.0 + e)) - 1.0

def fixed_target_grad(m):
    """Fixed form: -exp(-m) / (1 + exp(-m)). exp(-m) stays in the numerator,
    so it survives even when the denominator rounds to 1.0."""
    e = math.exp(-m)
    return -e / (1 + e)

for m in [5.0, 6.0, 16.0, 17.0, 40.0]:
    print(f"m={m:5.1f}  std fp32={std_target_grad_fp32(m): .3e}  "
          f"std bf16={std_target_grad_bf16(m): .3e}  fixed={fixed_target_grad(m): .3e}")

assert std_target_grad_fp32(17.0) == 0.0, "fp32 must collapse past the 16.6 cliff"
assert std_target_grad_bf16(6.0) == 0.0, "bf16 must collapse past the 5.5 cliff"
assert std_target_grad_fp32(16.0) != 0.0, "fp32 still alive just below its cliff"
assert abs(fixed_target_grad(17.0) - (-4.13993754739433e-08)) < 1e-15
print("OK: standard gradients hit exactly 0 past their cliffs; the fixed form does not.")

## Bug 1, the paper's own example: z = [9, -9, -9], target 0

Standard JAX/optax return loss 0.0 and target gradient 0.0. The fixed code returns loss 3.046e-8 and gradients that sum to zero.

In [ ]:
z = np.array([9.0, -9.0, -9.0], dtype=np.float32)
# Standard log-softmax (max-shift, fp32)
ls_std = z - z.max() - np.log(np.exp(z - z.max()).sum(dtype=np.float32))
print("standard log-softmax:", ls_std, " standard loss:", -float(ls_std[0]))
assert float(ls_std[0]) == 0.0 and -float(ls_std[0]) == 0.0

# Fixed: loss = softplus(-m), m = -logsumexp of non-target differences
m = -float(np.log(np.exp(z[1:] - z[0]).sum()))
fixed_loss = float(np.log1p(np.exp(-m)))
print(f"margin m = {m:.4f}; fixed loss = {fixed_loss:.4e}")
assert abs(fixed_loss - 3.046e-8) < 1e-10

# Fixed gradients: non-target e^(z_j - z_t)/(1+e^-m), target = negative sum (sums to 0 by construction)
g_non = np.exp(z[1:] - z[0]) / (1 + np.exp(-m))
g_fixed = np.array([-g_non.sum(), g_non[0], g_non[1]])
print("fixed gradients:", g_fixed, " sum:", float(g_fixed.sum()))
assert abs(float(g_fixed.sum())) < 1e-14
print("OK: fixed loss and gradients are non-zero and sum to zero; standard ones are exactly 0.")

## Bug 2: Softmax Cancellation — identical student and teacher, non-zero gradient

Standard code computes student probabilities with one formula (softmax) and teacher probabilities with another (exp of log-softmax). In fp32 the two formulas disagree in the last bits, so q - p is not zero even when the logits are identical. The paper's fix computes q - p directly from the logit difference, so equal logits give exactly zero.

In [ ]:
def softmax_fp32(v):
    v = np.asarray(v, dtype=np.float32)
    e = np.exp(v - v.max(), dtype=np.float32)
    return (e / e.sum(dtype=np.float32)).astype(np.float32)

def log_softmax_fp32(v):
    v = np.asarray(v, dtype=np.float32)
    return (v - v.max() - np.log(np.exp(v - v.max(), dtype=np.float32).sum(dtype=np.float32))).astype(np.float32)

zt = np.array([2.0, 0.5, -1.0], dtype=np.float32)
q = softmax_fp32(zt)                       # forward formula
p = np.exp(log_softmax_fp32(zt), dtype=np.float32).astype(np.float32)  # backward formula
floor = float(np.linalg.norm(q - p))
print("student == teacher logits, but ||q - p|| =", f"{floor:.3e}")
assert floor > 0, "the two formulas must disagree in fp32"

# Fixed path: same code path, same input -> identical output -> difference exactly 0
def fixed_soft_target_grad(delta):
    # gradient as a function of the logit DIFFERENCE only (paper Section 3 idea)
    ls_s = log_softmax_fp32(zt + np.float32(delta))
    ls_t = log_softmax_fp32(zt)
    return np.exp(ls_s) - np.exp(ls_t)  # delta = 0 -> identical arrays -> exactly 0

print("fixed gradient at delta = 0:", fixed_soft_target_grad(np.array([0.0, 0.0, 0.0], dtype=np.float32)))
assert np.all(fixed_soft_target_grad(np.array([0.0, 0.0, 0.0], dtype=np.float32)) == 0.0)
print("OK: standard path leaves a ~6e-8 noise floor at zero distance; the difference-based path is exactly 0.")

## The payoff: paper Table 1 (grokking, a+b mod 31, mean of 5 seeds, Adam eps = 1e-8)

Fixed code reaches 100% test accuracy in bf16 and fp32 where standard code stalls at 93.3% and 94.3%. The standard bf16 run stalls at a final margin of 5.1 — just under its 5.5 precision cliff — while the fixed run reaches 17.9.

In [ ]:
table = {"bf16 standard": 93.3, "bf16 fixed": 100.0,
         "fp32 standard": 94.3, "fp32 fixed": 100.0,
         "fp64 standard": 100.0, "fp64 fixed": 100.0}
for k, v in table.items():
    print(f"{k:15s} {v:.1f}%")
assert abs(table["bf16 fixed"] - table["bf16 standard"] - 6.7) < 1e-9
assert abs(table["fp32 fixed"] - table["fp32 standard"] - 5.7) < 1e-9
print("OK: +6.7pp in bf16, +5.7pp in fp32 at default Adam epsilon.")